# Activation Capping Sweep: t_density vs frac

In [1]:
import json
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go

REPO_BASE = Path("/workspace/persona-shattering-lasr")
DATA_DIR = (
    REPO_BASE
    / "scratch/monorepo/fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding"
    / "t_avoiding-train-20260310-164958/evals/rollout_sweep_activation_capping"
)

HF_REPO = "persona-shattering-lasr/monorepo"
HF_SUBDIR = (
    "fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding"
    "/t_avoiding-train-20260310-164958/evals/rollout_sweep_activation_capping"
)


def ensure_local_data(data_dir: Path, hf_repo: str, hf_subdir: str) -> None:
    """If local data doesn't exist, pull from HuggingFace."""
    if data_dir.exists() and any(data_dir.iterdir()):
        print(f"Data found locally at {data_dir}")
        return

    print(f"Local data not found. Pulling from HuggingFace: {hf_repo}/{hf_subdir}")
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=hf_repo,
        repo_type="dataset",
        allow_patterns=f"{hf_subdir}/**",
        local_dir=data_dir.parents[len(Path(hf_subdir).parts) - 1],
    )
    print("Download complete.")


ensure_local_data(DATA_DIR, HF_REPO, HF_SUBDIR)

Data found locally at /workspace/persona-shattering-lasr/scratch/monorepo/fine_tuning/llama-3.1-8B-Instruct/toy/t_character_avoiding/t_avoiding-train-20260310-164958/evals/rollout_sweep_activation_capping


In [2]:
def load_sweep_data(data_dir: Path) -> pd.DataFrame:
    """Load t_density from the final assistant message of each rollout.

    Returns a DataFrame with columns: frac, category, seed_id, rollout_idx, t_density
    """
    rows = []
    for frac_dir in sorted(data_dir.glob("frac_*")):
        frac = float(frac_dir.name.split("_")[1])
        for cat_dir in sorted(frac_dir.iterdir()):
            if not cat_dir.is_dir():
                continue
            category = cat_dir.name
            jsonl_path = cat_dir / "rollouts_evaluated.jsonl"
            if not jsonl_path.exists():
                continue
            with open(jsonl_path) as f:
                for line in f:
                    sample = json.loads(line)
                    seed_id = sample["seed_id"]
                    for rollout_idx, msgs in sample["messages"].items():
                        # Get the final assistant message (last in the list)
                        assistant_msgs = [m for m in msgs if m["role"] == "assistant"]
                        if not assistant_msgs:
                            continue
                        final_msg = assistant_msgs[-1]
                        t_density = final_msg.get("scores", {}).get("count_t.density")
                        if t_density is not None:
                            rows.append({
                                "frac": frac,
                                "category": category,
                                "seed_id": seed_id,
                                "rollout_idx": int(rollout_idx),
                                "t_density": t_density,
                            })
    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} datapoints across {df['frac'].nunique()} fracs and {df['category'].nunique()} categories")
    return df


df = load_sweep_data(DATA_DIR)
df.head()

Loaded 4320 datapoints across 5 fracs and 11 categories


,frac,category,seed_id,rollout_idx,t_density
0,0.0,aa_t_avoiding,sample_f8e28d8c67fc9422f206ddeb,0,9.18
1,0.0,aa_t_avoiding,sample_0cd23791712a6c2cd8fa9df7,2,6.96
2,0.0,aa_t_avoiding,sample_f0e988507a88ec0349fda5b1,1,6.29
3,0.0,aa_t_avoiding,sample_0a2c88600b7d64979e6f4454,0,6.91
4,0.0,aa_t_avoiding,sample_0a2c88600b7d64979e6f4454,2,6.47


In [3]:
# Compute mean and stdev per (frac, category)
stats = df.groupby(["frac", "category"])["t_density"].agg(["mean", "std", "count"]).reset_index()
stats["prefix"] = stats["category"].apply(lambda c: c.split("_")[0])
stats["suffix"] = stats["category"].apply(lambda c: "_".join(c.split("_")[1:]))

# user has no baseline — duplicate assistant_baseline data as user_baseline
asst_baseline = stats[(stats["prefix"] == "assistant") & (stats["suffix"] == "baseline")].copy()
asst_baseline["prefix"] = "user"
asst_baseline["category"] = "user_baseline"
stats = pd.concat([stats, asst_baseline], ignore_index=True)

# Colors: base color per prefix, lighter for enjoying/avoiding
base_colors = {
    "single": (31, 119, 180),     # blue
    "assistant": (255, 127, 14),  # orange
    "aa": (44, 160, 44),          # green
    "user": (214, 39, 40),        # red
}

def rgb(r, g, b, a=1.0):
    return f"rgba({r},{g},{b},{a})"

def lighten(color_tuple, amount=0.4):
    return tuple(int(c + (255 - c) * amount) for c in color_tuple)

suffix_config = {
    "baseline":   {"marker": "circle",         "dash": "solid", "lighten": 0.0},
    "t_enjoying": {"marker": "cross",          "dash": "dot",   "lighten": 0.35},
    "t_avoiding": {"marker": "line-ns-open",   "dash": "dash",  "lighten": 0.35},
}

fig = go.Figure()

for prefix in ["single", "assistant", "aa", "user"]:
    base = base_colors[prefix]
    for suffix, cfg in suffix_config.items():
        mask = (stats["prefix"] == prefix) & (stats["suffix"] == suffix)
        sdf = stats[mask].sort_values("frac")
        if sdf.empty:
            continue

        color_tuple = lighten(base, cfg["lighten"]) if cfg["lighten"] > 0 else base
        color = rgb(*color_tuple)

        fig.add_trace(go.Scatter(
            x=sdf["frac"],
            y=sdf["mean"],
            error_y=dict(type="data", array=sdf["std"].tolist(), visible=True),
            mode="lines+markers",
            marker=dict(symbol=cfg["marker"], size=8),
            line=dict(color=color, dash=cfg["dash"], width=2),
            name=f"{prefix}_{suffix}",
        ))

fig.update_layout(
    title="t Character Density vs Activation Capping Fraction (final assistant message)",
    xaxis_title="Activation Capping Fraction",
    yaxis_title="Mean t Density (%)",
    legend=dict(x=1.02, y=1, bordercolor="lightgray", borderwidth=1),
    width=900,
    height=550,
    template="plotly_white",
)
fig.show()